# 65 — LSEG/Gemma hybrid specificity filter

## Objective

Test one pre-outcome-frozen information-specificity filter around the existing
Notebook 36 hybrid: before Gemma firm-open aggregation, keep only explicit,
single-company, non-market-price/technical headlines. Filtered Gemma determines
the exact-event timing and leg counts; unchanged unfiltered FinBERT ranks the names.

The filter was frozen and pushed at commit `f26a1e5` after an input-only audit:
it retained all 7,348 firm-opens while raising defined Gemma strongest-event
firm-opens from 3,841 to 4,826. No return selected the filter.

This is a user-authorised iterative retrospective test on an opened eight-month
window. It cannot validate alpha, qualify deployment, replace the prospective
contract, or license another filter variant.


In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Markdown, display
from statsmodels.stats.multitest import multipletests

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "final_experiments":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final_experiments.lib.evaluate import annualized_sharpe  # noqa: E402
from final_experiments.lib.hybrid_specificity import (  # noqa: E402
    apply_high_specificity_filter,
    audit_topic_metadata,
    build_capped_hybrid_targets,
)
from final_experiments.lib.lseg_expanded import (  # noqa: E402
    assert_paired_population,
    build_expanded_aggregation_panel,
    canonical_score_frame,
    load_price_exports,
    load_success_scores,
    map_headlines_to_entry_sessions,
)
from final_experiments.lib.plots import CATEGORICAL, INK, apply_house_style  # noqa: E402
from final_experiments.lib.risk_overlay import paired_block_bootstrap_difference  # noqa: E402
from final_experiments.lib.sector_portfolios import SECTOR_MEMBERS  # noqa: E402
from final_experiments.lib.sparse_spread import ledger_rows_to_frame  # noqa: E402
from sentiment_benchmark.strategy_research.ledger import run_open_to_open_ledger  # noqa: E402
from sentiment_benchmark.strategy_research.market import OpenToOpenReturn  # noqa: E402
from sentiment_benchmark.strategy_research.portfolio import TargetPortfolio  # noqa: E402

COLLECTION_ROOT = REPO_ROOT / "Data/collections/lseg_us_sector_44_8m_headlines/derived"
GEMMA_SCORES = COLLECTION_ROOT / "headline_scores_gemma4_26b_a4b_it_deepinfra_fp8_investor_headline_soft_label_v1.csv"
FINBERT_SCORES = COLLECTION_ROOT / "headline_scores_finbert4556_cacheonly_20260803.csv"
MERGED_HEADLINES = COLLECTION_ROOT / "merged/headlines.jsonl"
PRICE_PATHS = (
    REPO_ROOT / "Data/derived/prices/lseg_us_sector_33_8m.csv",
    REPO_ROOT / "Data/derived/prices/lseg_us_sector_add11_8m.csv",
)
ORIGINAL33_PRICE_PATH = PRICE_PATHS[0]
SPEC_PATH = REPO_ROOT / "final_experiments/frozen_specs/lseg_gemma_hybrid_specificity_filter_v2.json"
N36_DIR = REPO_ROOT / "final_experiments/outputs/36_lseg_gemma_finbert_hybrid_alpha_audit"
N36_MANIFEST_PATH = N36_DIR / "manifest.json"
N36_DAILY_PATH = N36_DIR / "capped_hybrid_daily.parquet"
N36_FACTOR_PATH = N36_DIR / "factor_panel.csv"
OUTPUT_DIR = REPO_ROOT / "final_experiments/outputs/65_lseg_gemma_hybrid_specificity_filter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spec = json.loads(SPEC_PATH.read_text())
assert spec["status"] == "frozen_before_gemma_timing_specificity_portfolio_outcomes"
assert spec["experiment_id"] == "N65_lseg_gemma_hybrid_specificity_filter_v2"
SEED = int(spec["inference"]["seed"])
BLOCK_LENGTH = int(spec["inference"]["block_length_sessions"])
REPLICATIONS = int(spec["inference"]["replications"])
BH_Q = float(spec["inference"]["bh_q"])
PRIMARY_COST_BPS = float(spec["accounting"]["primary_cost_bps_per_side"])
COST_GRID = tuple(float(value) for value in spec["accounting"]["cost_curve_bps_per_side"])
INITIAL_NAV = float(spec["accounting"]["initial_nav_usd"])
SINGLE_NAME_CAP = float(spec["unchanged_strategy"]["single_name_cap"])
TOLERANCE = 1e-12

apply_house_style()
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
print("frozen experiment:", spec["experiment_id"])
print("classification: iterative retrospective; LSEG only; no source-regime pooling")
print("primary family:", spec["inference"]["primary_family"])


## Frozen plan

1. Verify that Reuters `subjects` and `entities` are unavailable rather than
   silently fabricating an event taxonomy.
2. Rebuild the paired 888,155-hash Gemma/FinBERT populations and apply exactly
   one return-blind filter to Gemma: explicit target, one company, not technical.
3. Reproduce Notebook 36 and retain its complete unfiltered FinBERT ranks.
4. Compare the filtered strategy with cash and the unchanged incumbent in one
   two-test BH family; also report the frozen sector-spanning HAC(5) intercept,
   chronological halves, cost curve, breadth, turnover, and tie resolution.
5. Stop regardless of outcome. There is no filter, threshold, cost, hedge,
   scaling, or holding-period sweep.


In [ ]:
topic_audit = audit_topic_metadata(MERGED_HEADLINES)
expected_input = spec["input_only_rationale"]
assert topic_audit["corpus_rows"] == expected_input["licensed_corpus_rows"]
assert topic_audit["rows_with_subjects"] == expected_input["rows_with_subjects"]
assert topic_audit["rows_with_entities"] == expected_input["rows_with_entities"]

gemma_raw, gemma_score_audit = load_success_scores(GEMMA_SCORES, scorer="gemma4_26b")
finbert_raw, finbert_score_audit = load_success_scores(FINBERT_SCORES, scorer="finbert")
paired = assert_paired_population(finbert_raw, gemma_raw)
canonical = {
    "gemma4_26b": canonical_score_frame(paired, scorer="gemma"),
    "finbert": canonical_score_frame(paired, scorer="finbert"),
}
filtered: dict[str, pd.DataFrame] = {}
filter_audits: dict[str, dict[str, object]] = {}
for scorer, frame in canonical.items():
    filtered[scorer], filter_audits[scorer] = apply_high_specificity_filter(frame)
if set(filtered["gemma4_26b"]["headline_sha256"]) != set(filtered["finbert"]["headline_sha256"]):
    raise RuntimeError("paired scorers did not retain the same filtered hashes")

prices44 = load_price_exports(*PRICE_PATHS)
panels: dict[tuple[str, str], pd.DataFrame] = {}
event_counts: dict[tuple[str, str], int] = {}
for variant, frames in (("unfiltered", canonical), ("specificity", filtered)):
    for scorer, frame in frames.items():
        events = map_headlines_to_entry_sessions(frame, prices44)
        event_counts[(variant, scorer)] = len(events)
        panels[(variant, scorer)] = build_expanded_aggregation_panel(events, prices44)

coverage_rows = []
for variant, frames in (("unfiltered", canonical), ("specificity", filtered)):
    for scorer, frame in frames.items():
        panel = panels[(variant, scorer)]
        coverage_rows.append(
            {
                "variant": variant,
                "scorer": scorer,
                "unique_headlines": frame["headline_sha256"].nunique(),
                "company_headline_associations": event_counts[(variant, scorer)],
                "firm_opens": len(panel),
                "defined_strongest_event_firm_opens": int(panel["strongest_event"].notna().sum()),
                "entry_sessions": panel["session_date"].nunique(),
                "companies": panel["symbol"].nunique(),
            }
        )
coverage = pd.DataFrame(coverage_rows)
gemma_coverage = coverage.loc[coverage["scorer"].eq("gemma4_26b")].set_index("variant")
assert gemma_coverage.loc["unfiltered", "company_headline_associations"] == expected_input["unfiltered_company_headline_associations"]
assert gemma_coverage.loc["specificity", "company_headline_associations"] == expected_input["filtered_company_headline_associations"]
assert gemma_coverage.loc["unfiltered", "firm_opens"] == expected_input["unfiltered_firm_opens"]
assert gemma_coverage.loc["specificity", "firm_opens"] == expected_input["filtered_firm_opens"]
assert (
    gemma_coverage.loc["unfiltered", "defined_strongest_event_firm_opens"]
    == expected_input["unfiltered_defined_gemma_strongest_event_firm_opens"]
)
assert (
    gemma_coverage.loc["specificity", "defined_strongest_event_firm_opens"]
    == expected_input["filtered_defined_gemma_strongest_event_firm_opens"]
)

topic_table = pd.DataFrame([topic_audit])
filter_table = pd.DataFrame(filter_audits).T.rename_axis("scorer").reset_index()
display(topic_table)
display(filter_table)
display(coverage)


## Portfolio construction and incumbent reproduction

The filtered and unfiltered panels now use the same immutable original-33
universe. Both call the tested helper that transfers Gemma's exact-extrema
counts to FinBERT's cross-sectional ranks. Before evaluating the new path, the
unfiltered reconstruction must reproduce Notebook 36 within `1e-12`.


In [ ]:
sector_members = {sector: tuple(members[:3]) for sector, members in SECTOR_MEMBERS.items()}
original_symbols = tuple(sorted(symbol for members in sector_members.values() for symbol in members))
target_sets: dict[str, tuple[TargetPortfolio, ...]] = {}
target_audits: dict[str, pd.DataFrame] = {}
for variant in ("unfiltered", "specificity"):
    gemma_panel = panels[(variant, "gemma4_26b")].loc[
        lambda frame: frame["symbol"].isin(original_symbols)
    ].copy()
    finbert_panel = panels[("unfiltered", "finbert")].loc[
        lambda frame: frame["symbol"].isin(original_symbols)
    ].copy()
    target_sets[variant], target_audits[variant] = build_capped_hybrid_targets(
        gemma_panel,
        finbert_panel,
        original_symbols,
        single_name_cap=SINGLE_NAME_CAP,
    )

sessions = pd.DatetimeIndex(pd.to_datetime(target_audits["unfiltered"]["session_date"]))
if not np.array_equal(sessions, pd.DatetimeIndex(pd.to_datetime(target_audits["specificity"]["session_date"]))):
    raise RuntimeError("filtered and unfiltered target schedules are not aligned")

prices = pd.read_csv(ORIGINAL33_PRICE_PATH, parse_dates=["session_date"])
prices["session_date"] = pd.to_datetime(prices["session_date"]).dt.normalize()
open_wide = prices.pivot(index="session_date", columns="symbol", values="open").sort_index().dropna()
locations = open_wide.index.get_indexer(sessions)
if (locations < 0).any() or not np.array_equal(locations[1:], locations[:-1] + 1):
    raise RuntimeError("signal sessions are not consecutive complete-price sessions")
if locations[-1] + 1 >= len(open_wide):
    raise RuntimeError("last signal session has no next open")

return_rows: list[OpenToOpenReturn] = []
for session, location in zip(sessions, locations, strict=True):
    next_session = open_wide.index[location + 1]
    returns = open_wide.loc[next_session, list(original_symbols)] / open_wide.loc[session, list(original_symbols)] - 1.0
    for symbol, value in returns.items():
        return_rows.append(
            OpenToOpenReturn(
                symbol=symbol,
                session=str(session.date()),
                next_session=str(next_session.date()),
                value=float(value),
            )
        )


def fold_final_liquidation(frame: pd.DataFrame) -> pd.DataFrame:
    liquidation = frame.loc[frame["final_liquidation"]]
    intervals = frame.loc[~frame["final_liquidation"]].copy().reset_index(drop=True)
    if len(liquidation) != 1 or intervals.empty:
        raise RuntimeError("expected one terminal liquidation")
    final = liquidation.iloc[0]
    last = intervals.index[-1]
    intervals.loc[last, "net_return"] = (1.0 + intervals.loc[last, "net_return"]) * (1.0 + final["net_return"]) - 1.0
    intervals.loc[last, "turnover"] += final["turnover"]
    intervals.loc[last, "cost"] += final["cost"]
    intervals.loc[last, "end_nav"] = final["end_nav"]
    return intervals


def run_targets(targets: tuple[TargetPortfolio, ...], cost_bps: float) -> pd.DataFrame:
    rows = run_open_to_open_ledger(
        targets,
        return_rows,
        cost_rate_per_side=cost_bps / 10_000.0,
        initial_nav_usd=INITIAL_NAV,
        force_final_liquidation=True,
    )
    return fold_final_liquidation(ledger_rows_to_frame(rows))


def summarize(daily: pd.DataFrame, cost_bps: float) -> dict[str, float | int]:
    gross = daily["gross_return"].to_numpy(dtype=float)
    net = daily["net_return"].to_numpy(dtype=float)
    equity = np.cumprod(1.0 + net)
    with_start = np.r_[1.0, equity]
    turnover = float(daily["turnover"].sum())
    return {
        "n_sessions": len(daily),
        "active_sessions": int(daily["gross_exposure"].gt(0).sum()),
        "mean_gross": float(gross.mean()),
        "mean_net": float(net.mean()),
        "sharpe_gross": float(annualized_sharpe(pd.Series(gross))),
        "sharpe_net": float(annualized_sharpe(pd.Series(net))),
        "ann_vol_net": float(np.std(net, ddof=1) * np.sqrt(252)),
        "total_return_gross": float(np.prod(1.0 + gross) - 1.0),
        "total_return_net": float(equity[-1] - 1.0),
        "max_drawdown": float(np.min(with_start / np.maximum.accumulate(with_start) - 1.0)),
        "mean_turnover": float(turnover / len(daily)),
        "total_turnover": turnover,
        "breakeven_bps_per_side": float(10_000.0 * gross.sum() / turnover),
        "cost_bps_per_side": cost_bps,
    }


daily_paths = {variant: run_targets(targets, PRIMARY_COST_BPS) for variant, targets in target_sets.items()}
n36_daily = pd.read_parquet(N36_DAILY_PATH).reset_index(drop=True)
n36_daily["session_date"] = pd.to_datetime(n36_daily["session_date"]).dt.normalize()
n36_manifest = json.loads(N36_MANIFEST_PATH.read_text())
if not np.array_equal(n36_daily["session_date"].to_numpy(), sessions.to_numpy()):
    raise RuntimeError("Notebook 36 daily path is not aligned")
reproduction_errors = {
    column: float(np.max(np.abs(daily_paths["unfiltered"][column] - n36_daily[column])))
    for column in ("gross_return", "net_return", "turnover", "gross_exposure")
}
if max(reproduction_errors.values()) > TOLERANCE:
    raise RuntimeError("raw-score unfiltered reconstruction does not reproduce Notebook 36")
if int(target_audits["unfiltered"]["eligible"].sum()) != n36_manifest["coverage"]["active_sessions"]:
    raise RuntimeError("unfiltered active-session count does not reproduce Notebook 36")

strategy_results = pd.DataFrame(
    [
        {"strategy": "notebook36_incumbent", **n36_manifest["result"]},
        {"strategy": "specificity_filtered_hybrid", **summarize(daily_paths["specificity"], PRIMARY_COST_BPS)},
    ]
)
cost_curve = pd.DataFrame(
    [
        {"strategy": "specificity_filtered_hybrid", **summarize(run_targets(target_sets["specificity"], cost), cost)}
        for cost in COST_GRID
    ]
)
identity = pd.DataFrame(
    [
        {
            "unfiltered_active_sessions": int(target_audits["unfiltered"]["eligible"].sum()),
            "filtered_active_sessions": int(target_audits["specificity"]["eligible"].sum()),
            "maximum_reproduction_error": max(reproduction_errors.values()),
            "filtered_max_abs_name_weight": float(target_audits["specificity"]["max_abs_name_weight"].max()),
        }
    ]
)
display(identity.T)
display(strategy_results)
display(cost_curve[["cost_bps_per_side", "sharpe_net", "total_return_net", "breakeven_bps_per_side"]])


## Inference, factors, and temporal stability

The primary family contains only the two comparisons frozen in the pushed
specification. The factor intercept and two chronological halves are reported
under their separate predeclared roles; none is used to create another variant.


In [ ]:
filtered_daily = daily_paths["specificity"]
cash = filtered_daily[["session_date"]].copy()
cash["net_return"] = 0.0
comparison_specs = [
    ("specificity_filtered_hybrid_minus_cash", filtered_daily, cash),
    ("specificity_filtered_hybrid_minus_notebook36_incumbent", filtered_daily, n36_daily),
]
inference_rows = []
for comparison, candidate, reference in comparison_specs:
    stats = paired_block_bootstrap_difference(
        candidate[["session_date", "net_return"]],
        reference[["session_date", "net_return"]],
        value_col="net_return",
        block_length=BLOCK_LENGTH,
        replications=REPLICATIONS,
        seed=SEED,
    )
    inference_rows.append({"comparison": comparison, **stats})
primary_inference = pd.DataFrame(inference_rows)
primary_inference["mean_difference_bps_session"] = primary_inference["mean_difference"] * 10_000
primary_inference["ci_low_bps_session"] = primary_inference["ci_low"] * 10_000
primary_inference["ci_high_bps_session"] = primary_inference["ci_high"] * 10_000
reject, q_values, _, _ = multipletests(primary_inference["p_two_sided"], alpha=BH_Q, method="fdr_bh")
primary_inference["bh_q_value"] = q_values
primary_inference["bh_reject"] = reject
primary_inference["positive_gate"] = (
    primary_inference["mean_difference"].gt(0)
    & primary_inference["ci_low"].gt(0)
    & primary_inference["bh_reject"]
)

factor_panel = pd.read_csv(N36_FACTOR_PATH, parse_dates=["session_date"])
factor_panel["session_date"] = pd.to_datetime(factor_panel["session_date"]).dt.normalize()
factor_columns = ["equal_weight_market", *[column for column in factor_panel if column.startswith("spread_")]]
regression = filtered_daily[["session_date", "net_return"]].merge(factor_panel, on="session_date", validate="1:1")
design = sm.add_constant(regression[factor_columns], has_constant="add")
if np.linalg.matrix_rank(design.to_numpy(dtype=float)) != design.shape[1]:
    raise RuntimeError("factor design is rank deficient")
fit = sm.OLS(regression["net_return"], design).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": BLOCK_LENGTH, "use_correction": True},
)
interval = fit.conf_int(alpha=0.05).loc["const"]
factor_model = pd.DataFrame(
    [
        {
            "strategy": "specificity_filtered_hybrid",
            "n_sessions": int(fit.nobs),
            "alpha_bps_session": float(fit.params["const"] * 10_000),
            "alpha_ci_low_bps_session": float(interval.iloc[0] * 10_000),
            "alpha_ci_high_bps_session": float(interval.iloc[1] * 10_000),
            "alpha_t_hac5": float(fit.tvalues["const"]),
            "alpha_p_two_sided_hac5": float(fit.pvalues["const"]),
            "r_squared": float(fit.rsquared),
            "market_beta": float(fit.params["equal_weight_market"]),
        }
    ]
)
factor_model["positive_gate"] = (
    factor_model["alpha_bps_session"].gt(0)
    & factor_model["alpha_ci_low_bps_session"].gt(0)
    & factor_model["alpha_p_two_sided_hac5"].lt(0.05)
)

split = (len(sessions) + 1) // 2
temporal_halves = pd.DataFrame(
    [
        {
            "strategy": "specificity_filtered_hybrid",
            "period": period,
            "sessions": len(values),
            "mean_net_bps_session": float(values.mean() * 10_000),
            "sharpe_net": float(annualized_sharpe(values)),
            "total_return_net": float(np.prod(1.0 + values) - 1.0),
        }
        for period, values in (
            ("first_half", filtered_daily["net_return"].iloc[:split]),
            ("second_half", filtered_daily["net_return"].iloc[split:]),
        )
    ]
)

primary_index = primary_inference.set_index("comparison")
filtered_result = strategy_results.set_index("strategy").loc["specificity_filtered_hybrid"]
cash_pass = bool(primary_index.loc["specificity_filtered_hybrid_minus_cash", "positive_gate"])
improvement_pass = bool(primary_index.loc["specificity_filtered_hybrid_minus_notebook36_incumbent", "positive_gate"])
factor_pass = bool(factor_model.loc[0, "positive_gate"])
both_halves_positive = bool(temporal_halves["mean_net_bps_session"].gt(0).all())
break_even_pass = bool(filtered_result["breakeven_bps_per_side"] > PRIMARY_COST_BPS)
active_pass = bool(filtered_result["active_sessions"] >= 60)
weight_pass = bool(identity.loc[0, "filtered_max_abs_name_weight"] <= SINGLE_NAME_CAP + TOLERANCE)
if cash_pass and improvement_pass and factor_pass and both_halves_positive and break_even_pass and active_pass and weight_pass:
    result_class = "retrospective_improvement"
elif cash_pass and both_halves_positive:
    result_class = "costed_positive_not_improved"
else:
    result_class = "nonviable"
classification = pd.DataFrame(
    [
        {
            "strategy": "specificity_filtered_hybrid",
            "classification": result_class,
            "cash_gate": cash_pass,
            "incumbent_improvement_gate": improvement_pass,
            "factor_gate": factor_pass,
            "both_half_means_positive": both_halves_positive,
            "breakeven_gate": break_even_pass,
            "active_session_gate": active_pass,
            "single_name_cap_gate": weight_pass,
        }
    ]
)
primary_columns = [
    "comparison",
    "mean_difference_bps_session",
    "ci_low_bps_session",
    "ci_high_bps_session",
    "p_two_sided",
    "bh_q_value",
    "positive_gate",
]
display(primary_inference[primary_columns])
display(factor_model)
display(temporal_halves)
display(classification.T)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.0))
for label, daily, color in (
    ("Notebook 36 incumbent", n36_daily, CATEGORICAL[0]),
    ("Specificity-filtered hybrid", filtered_daily, CATEGORICAL[1]),
):
    axes[0, 0].plot(
        daily["return_end_date"],
        np.cumprod(1.0 + daily["net_return"]),
        lw=2.0,
        label=label,
        color=color,
    )
axes[0, 0].axhline(1.0, color=INK["reference"], lw=0.8)
axes[0, 0].set_ylabel("Growth of $1")
axes[0, 0].set_title("After-cost paths at 10 bps/side")
axes[0, 0].legend(frameon=False)

axes[0, 1].plot(cost_curve["cost_bps_per_side"], cost_curve["sharpe_net"], marker="o", lw=2.0, color=CATEGORICAL[1])
axes[0, 1].axhline(0.0, color=INK["reference"], lw=0.8)
axes[0, 1].axvline(PRIMARY_COST_BPS, color=INK["reference"], lw=0.8, ls="--")
axes[0, 1].set_xlabel("Cost (bps per side)")
axes[0, 1].set_ylabel("Net Sharpe")
axes[0, 1].set_title("Frozen cost curve")

x = np.arange(len(primary_inference))
means = primary_inference["mean_difference_bps_session"].to_numpy(dtype=float)
lower = means - primary_inference["ci_low_bps_session"].to_numpy(dtype=float)
upper = primary_inference["ci_high_bps_session"].to_numpy(dtype=float) - means
axes[1, 0].bar(x, means, color=[CATEGORICAL[1], CATEGORICAL[3]], alpha=0.85)
axes[1, 0].errorbar(x, means, yerr=np.vstack([lower, upper]), fmt="none", color=INK["reference"], capsize=4)
axes[1, 0].axhline(0.0, color=INK["reference"], lw=0.8)
axes[1, 0].set_xticks(x, ["Filtered−cash", "Filtered−incumbent"], rotation=10)
axes[1, 0].set_ylabel("Net difference (bps/session)")
axes[1, 0].set_title("Frozen two-test family")

resolution = coverage.loc[coverage["scorer"].eq("gemma4_26b")].set_index("variant")
bars = pd.DataFrame(
    {
        "Defined firm-opens": resolution["defined_strongest_event_firm_opens"],
        "Active strategy sessions": [
            int(target_audits[variant]["eligible"].sum()) for variant in resolution.index
        ],
    },
    index=resolution.index,
)
bars.plot.bar(ax=axes[1, 1], color=[CATEGORICAL[2], CATEGORICAL[4]], rot=0)
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Input resolution and executable activity")
axes[1, 1].legend(frameon=False, fontsize=8)

fig.suptitle("Iterative LSEG/Gemma information-specificity test")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "specificity_filter_results.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
topic_table.to_csv(OUTPUT_DIR / "topic_metadata_audit.csv", index=False)
filter_table.to_csv(OUTPUT_DIR / "filter_audit.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "coverage.csv", index=False)
identity.to_csv(OUTPUT_DIR / "identity.csv", index=False)
strategy_results.to_csv(OUTPUT_DIR / "strategy_results.csv", index=False)
cost_curve.to_csv(OUTPUT_DIR / "cost_curve.csv", index=False)
primary_inference.to_csv(OUTPUT_DIR / "primary_inference.csv", index=False)
factor_model.to_csv(OUTPUT_DIR / "factor_model.csv", index=False)
temporal_halves.to_csv(OUTPUT_DIR / "temporal_halves.csv", index=False)
classification.to_csv(OUTPUT_DIR / "classification.csv", index=False)
for variant, daily in daily_paths.items():
    daily.to_parquet(OUTPUT_DIR / f"{variant}_daily.parquet", index=False)
    target_audits[variant].to_parquet(OUTPUT_DIR / f"{variant}_target_audit.parquet", index=False)

try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None

result_index = strategy_results.set_index("strategy")
incumbent = result_index.loc["notebook36_incumbent"]
filtered_result = result_index.loc["specificity_filtered_hybrid"]
cash_row = primary_index.loc["specificity_filtered_hybrid_minus_cash"]
improvement_row = primary_index.loc["specificity_filtered_hybrid_minus_notebook36_incumbent"]
decision = {
    "classification": result_class,
    "incumbent_net_sharpe": float(incumbent["sharpe_net"]),
    "filtered_net_sharpe": float(filtered_result["sharpe_net"]),
    "incumbent_total_return_net": float(incumbent["total_return_net"]),
    "filtered_total_return_net": float(filtered_result["total_return_net"]),
    "cash_bh_q": float(cash_row["bh_q_value"]),
    "incumbent_improvement_bh_q": float(improvement_row["bh_q_value"]),
    "factor_alpha_p": float(factor_model.loc[0, "alpha_p_two_sided_hac5"]),
    "prospective_contract_changed": False,
    "validated_alpha": False,
    "deployment_qualified": False,
    "further_filter_search_permitted": False,
}
manifest = {
    "status": "ITERATIVE_RETROSPECTIVE_LSEG_GEMMA_TIMING_SPECIFICITY_FILTER",
    "notebook": "65_lseg_gemma_hybrid_specificity_filter.ipynb",
    "git_commit_at_execution": git_commit,
    "frozen_spec_sha256": hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest(),
    "specification": spec,
    "inputs": {
        "gemma_score_output_sha256": gemma_score_audit["output_sha256"],
        "finbert_score_output_sha256": finbert_score_audit["output_sha256"],
        "merged_headlines_path": str(MERGED_HEADLINES.relative_to(REPO_ROOT)),
        "notebook36_manifest_sha256": hashlib.sha256(N36_MANIFEST_PATH.read_bytes()).hexdigest(),
        "licensed_headline_text_loaded": False,
        "source_regimes_pooled": False,
    },
    "topic_metadata_audit": topic_audit,
    "filter_audits": filter_audits,
    "coverage": json.loads(coverage.to_json(orient="records")),
    "identity": json.loads(identity.to_json(orient="records"))[0],
    "strategy_results": json.loads(strategy_results.to_json(orient="records")),
    "cost_curve": json.loads(cost_curve.to_json(orient="records")),
    "primary_inference": json.loads(primary_inference.to_json(orient="records")),
    "factor_model": json.loads(factor_model.to_json(orient="records"))[0],
    "temporal_halves": json.loads(temporal_halves.to_json(orient="records")),
    "classification": json.loads(classification.to_json(orient="records"))[0],
    "decision": decision,
    "limitations": [
        "The eight-month LSEG return window and incumbent hybrid were opened before this filter was proposed.",
        "The deterministic relevance metadata is not human validated and Reuters subject/entity fields are empty.",
        "This is one user-authorised retrospective filter test, not independent validation.",
        "Daily bars omit bid-ask spreads, auction slippage, financing, borrow constraints, taxes, and market impact.",
        "Split-adjusted price returns exclude dividends.",
        "No outcome may change the frozen prospective strategy or license another historical filter.",
    ],
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

display(
    Markdown(
        "### Decision\n\n"
        f"- Specificity-filtered hybrid: net Sharpe **{filtered_result['sharpe_net']:.3f}**, "
        f"net return **{filtered_result['total_return_net']:+.2%}**, break-even "
        f"**{filtered_result['breakeven_bps_per_side']:.2f} bps/side**, class "
        f"**{result_class}**.\n"
        f"- Notebook 36 incumbent: net Sharpe **{incumbent['sharpe_net']:.3f}**, "
        f"net return **{incumbent['total_return_net']:+.2%}**.\n"
        f"- Filtered minus cash BH q=**{cash_row['bh_q_value']:.4f}**; filtered minus "
        f"incumbent BH q=**{improvement_row['bh_q_value']:.4f}**; sector-spanning "
        f"alpha p=**{factor_model.loc[0, 'alpha_p_two_sided_hac5']:.4f}**.\n"
        "- This opened-window result does not validate alpha, qualify deployment, "
        "change the prospective contract, or permit another filter variant."
    )
)
print(json.dumps(decision, indent=2))


## Next step

Refresh the aggregate-only strategy synthesis and result documents with the
frozen classification. The only independent alpha evidence remains the
prospective new-date replay after its population and activity gates pass.
